In [ ]:
import sys
import os
import numpy as np
from pathlib import Path
from matplotlib import pyplot as plt

In [ ]:
from importlib import reload

sys.path.append(os.path.abspath("../"))

import Evaluation.evaluation_util
reload(Evaluation.evaluation_util)

EVAL_DIR = Path(".")

LOSS_CONFIG_KEYS = [
    "bce_loss_weight", "cl_dice_loss_weight", "dice_loss_weight", "focal_loss_weight",
    "junction_heatmap_weight_scale", "junction_patch_weight",
    "skeleton_recall_loss_weight", "topological_loss_weight",
]

METRIC_COLS = [
    "Dice", "IoU", "clDice", "tprec", "tsens",
    "Dice Postprocessed", "IoU Postprocessed",
    "clDice Postprocessed", "tprec Postprocessed", "tsens Postprocessed",
    "Betti0 F-Score@0.5", "Betti1 MAE@0.5",
]

In [ ]:
# Load metrics and Betti CSVs produced by compute_metrics.py
df = Evaluation.evaluation_util.load_latest_metrics_csv(EVAL_DIR)
df_betti = Evaluation.evaluation_util.load_latest_betti_csv(EVAL_DIR)

if df.empty:
    print("No metrics CSV found – run compute_metrics.py first.")
else:
    print(f"Loaded metrics for {len(df)} model(s).")

if df_betti.empty:
    print("No Betti CSV found – Betti visualisations will be skipped.")
else:
    print(f"Loaded Betti data: {len(df_betti)} rows.")

In [ ]:
## Metrics table – best value per column is bold
present_metric_cols = [c for c in METRIC_COLS if c in df.columns]
present_loss_cols = [c for c in LOSS_CONFIG_KEYS if c in df.columns]

# Higher is better for all metrics except MAE
higher_is_better = {c: "Betti1 MAE" not in c for c in present_metric_cols}


def highlight_best(s):
    if s.name not in present_metric_cols:
        return [""] * len(s)
    want_max = higher_is_better.get(s.name, True)
    best_val = s.max() if want_max else s.min()
    return ["font-weight: bold" if v == best_val else "" for v in s]


display_cols = present_metric_cols + present_loss_cols
df[display_cols].style.apply(highlight_best, axis=0).format(
    {col: "{:.4f}" for col in present_metric_cols}
)

In [ ]:
## Bar charts for segmentation metrics
bar_metrics = [
    ("Dice", "IoU", "clDice"),
    ("tprec", "tsens"),
    ("Dice Postprocessed", "IoU Postprocessed", "clDice Postprocessed"),
    ("tprec Postprocessed", "tsens Postprocessed"),
]

# Short model labels for readability
short_labels = [name[-15:] for name in df.index]

for group in bar_metrics:
    cols = [c for c in group if c in df.columns]
    if not cols:
        continue
    fig, axes = plt.subplots(1, len(cols), figsize=(5 * len(cols), max(5, len(df) * 0.4 + 2)),
                             sharey=True)
    if len(cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        values = df[col].astype(float)
        bars = ax.barh(short_labels, values, color="steelblue")
        best_idx = values.idxmax() if higher_is_better.get(col, True) else values.idxmin()
        best_pos = list(df.index).index(best_idx)
        bars[best_pos].set_color("darkorange")
        ax.set_xlim(max(0, values.min() - 0.05), min(1.0, values.max() + 0.05))
        ax.set_xlabel(col)
        ax.axvline(x=values.mean(), color="gray", linestyle="--", linewidth=1, label="mean")
        ax.legend(fontsize=8)
    plt.suptitle(" / ".join(cols), fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
## Betti-0 F-Score and Betti-1 MAE bar chart (summary at threshold 0.5)
betti_summary_cols = [c for c in ["Betti0 F-Score@0.5", "Betti1 MAE@0.5"] if c in df.columns]
if betti_summary_cols:
    fig, axes = plt.subplots(1, len(betti_summary_cols),
                             figsize=(5 * len(betti_summary_cols), max(5, len(df) * 0.4 + 2)),
                             sharey=True)
    if len(betti_summary_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, betti_summary_cols):
        values = df[col].astype(float)
        want_max = "MAE" not in col
        bars = ax.barh(short_labels, values, color="mediumseagreen")
        best_idx = values.idxmax() if want_max else values.idxmin()
        best_pos = list(df.index).index(best_idx)
        bars[best_pos].set_color("darkorange")
        ax.set_xlabel(col)
    plt.suptitle("Topological metrics (threshold = 0.5)", fontsize=12)
    plt.tight_layout()
    plt.show()
    plt.close()
else:
    print("No Betti summary columns found in metrics CSV.")

In [ ]:
## Betti curves per model (averaged over all images), overlaid on a single plot
# Requires the betti_*.csv produced by compute_metrics.py
import re as _re

if df_betti.empty:
    print("No Betti CSV available – skipping curve plot.")
else:
    BETTI_THRESHOLDS = np.linspace(0.0, 1.0, 11)

    fig, ax_f = plt.subplots(figsize=(10, 5))
    ax_e = ax_f.twinx()
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for i, model_name in enumerate(df.index):
        df_model = df_betti[
            (df_betti["model"] == model_name) & (df_betti["type"] == "predicted")
        ]
        if df_model.empty:
            continue

        b0_cols = [f"b0_t{t:.2f}" for t in BETTI_THRESHOLDS]
        b1_cols = [f"b1_t{t:.2f}" for t in BETTI_THRESHOLDS]

        # Filter to columns that actually exist
        b0_cols = [c for c in b0_cols if c in df_model.columns]
        b1_cols = [c for c in b1_cols if c in df_model.columns]
        thresholds_present = np.array([float(c.split("_t")[1]) for c in b0_cols])

        # Groundtruth for this model
        df_gt = df_betti[
            (df_betti["model"] == model_name) & (df_betti["type"] == "groundtruth")
        ]
        gt_b0 = df_gt[b0_cols].values.mean(axis=0) if not df_gt.empty else None
        gt_b1 = df_gt[b1_cols].values.mean(axis=0) if not df_gt.empty else None

        # Per-image curves → aggregate
        pred_b0_curves = df_model[b0_cols].values.tolist()
        pred_b1_curves = df_model[b1_cols].values.tolist()
        gt_b0_list = df_gt[b0_cols].values[:, 0].tolist() if gt_b0 is not None else []
        gt_b1_list = df_gt[b1_cols].values[:, 0].tolist() if gt_b1 is not None else []

        fscore_per_t, mae_b1_per_t = compute_betti_aggregate_metrics(
            pred_b0_curves, pred_b1_curves, gt_b0_list, gt_b1_list, thresholds_present
        )

        short = model_name[-20:]
        color = colors[i % len(colors)]
        ax_f.plot(thresholds_present, fscore_per_t, color=color,
                  linewidth=2, label=short)
        ax_e.plot(thresholds_present, mae_b1_per_t, color=color,
                  linewidth=1.5, linestyle="--")

    ax_f.set_xlabel("Probability Threshold")
    ax_f.set_ylabel("Avg. Betti-0 F-Score (↑)")
    ax_e.set_ylabel("Avg. Betti-1 MAE (↓, dashed)")
    ax_f.invert_xaxis()
    ax_f.legend(loc="upper left", fontsize=8)
    plt.title("Topological Evaluation Profile per Model")
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
## Scatter: Dice Postprocessed vs Betti0 F-Score@0.5
if "Dice Postprocessed" in df.columns and "Betti0 F-Score@0.5" in df.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    for i, (model_name, row) in enumerate(df.iterrows()):
        x = float(row["Dice Postprocessed"])
        y = float(row["Betti0 F-Score@0.5"])
        ax.scatter(x, y, color=colors[i % len(colors)], s=80, zorder=3)
        ax.annotate(model_name[-18:], (x, y), textcoords="offset points",
                    xytext=(5, 3), fontsize=7)
    ax.set_xlabel("Dice Postprocessed (↑)")
    ax.set_ylabel("Betti-0 F-Score @ 0.5 (↑)")
    ax.set_title("Segmentation quality vs. Topological accuracy")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close()
else:
    print("Columns for scatter plot not available – skipping.")